#ML MODEL FOR Protein–Nanoparticle Interaction Prediction (Nef Binding ML Model)

### Step 1: Install necessary libraries and download datasets



In [1]:
# Install kagglehub and biopython
!pip install kagglehub biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.2 MB/s eta 0:00:00


In [2]:
import kagglehub
import pandas as pd

# Download the nanoparticle toxicity dataset
toxicity_path = kagglehub.dataset_download("ucimachinelearning/nanoparticle-toxicity-dataset")
print(f"Path to nanoparticle toxicity dataset files: {toxicity_path}")

# Download the PPI dataset
ppi_path = kagglehub.dataset_download("spandansureja/ppi-dataset")
print(f"Path to PPI dataset files: {ppi_path}")


100%|██████████| 2.75k/2.75k [00:00<00:00, 5.62MB/s]

Extracting files...
Path to nanoparticle toxicity dataset files: /root/.cache/kagglehub/datasets/ucimachinelearning/nanoparticle-toxicity-dataset/versions/1


100%|██████████| 47.7M/47.7M [00:00<00:00, 80.8MB/s]

Extracting files...


Path to PPI dataset files: /root/.cache/kagglehub/datasets/spandansureja/ppi-dataset/versions/1


### Step 2: Load and Inspect Datasets



In [3]:
import kagglehub
import pandas as pd
import os
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Seq import Seq

# --- Step 1: Download and Load Nanoparticle Toxicity Dataset ---
# Download the nanoparticle toxicity dataset
toxicity_path = kagglehub.dataset_download("ucimachinelearning/nanoparticle-toxicity-dataset")
print(f"Path to nanoparticle toxicity dataset files: {toxicity_path}")

# List files in the toxicity dataset directory to confirm file names
toxicity_files = os.listdir(toxicity_path)
print(f"Files in Nanoparticle Toxicity dataset directory: {toxicity_files}")

# Load Nanoparticle Toxicity Dataset with the correct file name
toxicity_df = pd.read_csv(f"{toxicity_path}/nanotox_dataset.csv")
print("\nNanoparticle Toxicity DataFrame Head:")
display(toxicity_df.head())
print("\nNanoparticle Toxicity DataFrame Info:")
toxicity_df.info()

# --- Step 2: Download and Load PPI Dataset ---
# Download the PPI dataset
ppi_path = kagglehub.dataset_download("spandansureja/ppi-dataset")
print(f"Path to PPI dataset files: {ppi_path}")

# Load PPI Dataset
# The PPI dataset might contain multiple files. Let's try to find the main one.
# A common practice is to look for a CSV with 'protein' or 'sequence' in its name, or simply the largest CSV.
ppi_files = os.listdir(ppi_path)
print(f"Files in PPI dataset directory: {ppi_files}")

# Assuming the main data is in 'pep_seq.csv' or similar
if 'pep_seq.csv' in ppi_files:
    ppi_df = pd.read_csv(f"{ppi_path}/pep_seq.csv")
elif 'protein_sequences.csv' in ppi_files:
    ppi_df = pd.read_csv(f"{ppi_path}/protein_sequences.csv")
elif 'positive_protein_sequences.csv' in ppi_files:
    ppi_df = pd.read_csv(f"{ppi_path}/positive_protein_sequences.csv")
else:
    # Try to find a suitable CSV, or ask the user if ambiguous
    csv_files = [f for f in ppi_files if f.endswith('.csv')]
    if csv_files:
        print(f"Found CSV files: {csv_files}. Loading the first one: {csv_files[0]}")
        ppi_df = pd.read_csv(f"{ppi_path}/{csv_files[0]}")
    else:
        raise FileNotFoundError("No suitable CSV file found in the PPI dataset directory.")

print("\nPPI DataFrame Head:")
display(ppi_df.head())
print("\nPPI DataFrame Info:")
ppi_df.info()

# --- Step 3: Generate Protein Physicochemical Descriptors ---
def get_protein_features(sequence):
    if not isinstance(sequence, str) or not sequence:
        return None, None, None, None
    try:
        # Ensure the sequence is a Biopython Seq object
        protein_seq = Seq(sequence)
        analysed_seq = ProteinAnalysis(str(protein_seq))

        # Isoelectric Point (pI)
        pI = analysed_seq.isoelectric_point()

        # GRAVY (Grand Average of Hydropathy)
        gravy = analysed_seq.gravy()

        # Molecular Weight (MW)
        mw = analysed_seq.molecular_weight()

        # Charge at pH 7.4 (approximate blood pH)
        if pI > 7.4:
            charge_at_7_4 = 'Positive' # More basic, so more positive charge at physiological pH
        elif pI < 7.4:
            charge_at_7_4 = 'Negative' # More acidic, so more negative charge at physiological pH
        else:
            charge_at_7_4 = 'Neutral'

        return pI, gravy, mw, charge_at_7_4
    except Exception as e:
        print(f"Error processing sequence {sequence[:20]}...: {e}")
        return None, None, None, None

# Get all unique protein sequences from both columns of ppi_df
unique_protein_sequences = pd.concat([ppi_df['protein_sequences_1'], ppi_df['protein_sequences_2']]).unique()

# Create a DataFrame to store protein features
protein_features = []
for seq in unique_protein_sequences:
    pI, gravy, mw, charge_7_4 = get_protein_features(seq)
    protein_features.append({'sequence': seq, 'pI': pI, 'GRAVY': gravy, 'MW': mw, 'Charge_at_pH_7_4': charge_7_4})

protein_features_df = pd.DataFrame(protein_features)
display(protein_features_df.head())
protein_features_df.info()

# --- Step 4: Combine Datasets, Add Environmental Variables, and Generate Target Variable ---
# Clean protein_features_df by dropping rows with None values (failed calculations)
# and rename 'sequence' for clarity
protein_features_df_cleaned = protein_features_df.dropna(subset=['pI', 'GRAVY', 'MW', 'Charge_at_pH_7_4']).copy()
protein_features_df_cleaned = protein_features_df_cleaned.rename(columns={'sequence': 'protein_sequence'})

# To create a Cartesian product of nanoparticles and proteins, we'll add a dummy key to both dataframes
toxicity_df_temp = toxicity_df.copy()
protein_features_df_temp = protein_features_df_cleaned.copy()

toxicity_df_temp['key'] = 1
protein_features_df_temp['key'] = 1

# Perform a merge on the dummy key to get all combinations
interaction_df = pd.merge(toxicity_df_temp, protein_features_df_temp, on='key').drop('key', axis=1)

# Add Environmental Conditions (Standard Lab Constants)
# These are illustrative constant values as requested.
interaction_df['Medium_pH'] = 7.4
interaction_df['Ionic_Strength'] = 0.15 # Molar (e.g., for PBS)
interaction_df['Temperature_C'] = 25.0 # Celsius
interaction_df['Temperature_K'] = 298.15 # Kelvin

# Generate Target Variable ('Interaction' Binary) based on the specific logic:
# "If Nanoparticle Zeta Potential is Positive (from the Toxicity dataset)
# AND Protein pI is < 6 (meaning it's Negative at blood pH) THEN set the missing 'Interaction' label to 1."

# Initialize 'Interaction' column to 0 (no interaction by default)
interaction_df['Interaction'] = 0

# Apply the specific rule: if nanoparticle surfcharge > 0 and protein pI < 6, set Interaction to 1
interaction_df.loc[(interaction_df['surfcharge'] > 0) & (interaction_df['pI'] < 6), 'Interaction'] = 1

print("Combined Interaction DataFrame Head:")
display(interaction_df.head())
print("\nCombined Interaction DataFrame Info:")
interaction_df.info()

Using Colab cache for faster access to the 'nanoparticle-toxicity-dataset' dataset.
Path to nanoparticle toxicity dataset files: /kaggle/input/nanoparticle-toxicity-dataset
Files in Nanoparticle Toxicity dataset directory: ['nanotox_dataset.csv', '.nfs000000001c7d55af00000074']

Nanoparticle Toxicity DataFrame Head:


,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,class
0,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,nonToxic
1,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.010,1.61,3,nonToxic
2,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.100,1.61,3,nonToxic
3,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,1.000,1.61,3,nonToxic
4,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,5.000,1.61,3,nonToxic



Nanoparticle Toxicity DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 881 entries, 0 to 880
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   NPs         881 non-null    object 
 1   coresize    881 non-null    float64
 2   hydrosize   881 non-null    float64
 3   surfcharge  881 non-null    float64
 4   surfarea    881 non-null    float64
 5   Ec          881 non-null    float64
 6   Expotime    881 non-null    int64  
 7   dosage      881 non-null    float64
 8   e           881 non-null    float64
 9   NOxygen     881 non-null    int64  
 10  class       881 non-null    object 
dtypes: float64(7), int64(2), object(2)
memory usage: 75.8+ KB
Using Colab cache for faster access to the 'ppi-dataset' dataset.
Path to PPI dataset files: /kaggle/input/ppi-dataset
Files in PPI dataset directory: ['negative_protein_sequences.csv', 'positive_protein_sequences.csv', '.nfs00000000203ce43100000076']

PPI Dat

,protein_sequences_1,protein_sequences_2
0,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,MARPHPWWLCVLGTLVGLSATPAPKSCPERHYWAQGKLCCQMCEPG...
1,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,MAENVVEPGPPSAKRPKLSSPALSASASDGTDFGSLFDLEHDLPDE...
2,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,MEGGRRARVVIESKRNFFLGAFPTPFPAEHVELGRLGDSETAMVPG...
3,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,MLFYSFFKSLVGKDVVVELKNDLSICGTLHSVDQYLNIKLTDISVT...
4,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,MQTIKCVVVGDGAVGKTCLLISYTTNKFPSEYVPTVFDNYAVTVMI...



PPI DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36630 entries, 0 to 36629
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   protein_sequences_1  36630 non-null  object
 1   protein_sequences_2  36630 non-null  object
dtypes: object(2)
memory usage: 572.5+ KB
Error processing sequence MERQEESLSARPALETEGLR...: 'U'
Error processing sequence MGCAEGKAVAAAAPTELQTK...: 'U'
Error processing sequence MCAARLAAAAAAAQSVYAFS...: 'U'


,sequence,pI,GRAVY,MW,Charge_at_pH_7_4
0,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,8.231468,-0.486796,64489.4386,Positive
1,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,9.438832,-0.865882,27884.9954,Positive
2,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,5.557739,-0.862530,48197.1750,Negative
3,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,8.676558,-0.606215,100829.9852,Positive
4,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,5.265132,-0.157192,250533.5881,Negative


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9456 entries, 0 to 9455
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sequence          9456 non-null   object 
 1   pI                9453 non-null   float64
 2   GRAVY             9453 non-null   float64
 3   MW                9453 non-null   float64
 4   Charge_at_pH_7_4  9453 non-null   object 
dtypes: float64(3), object(2)
memory usage: 369.5+ KB
Combined Interaction DataFrame Head:


,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,...,protein_sequence,pI,GRAVY,MW,Charge_at_pH_7_4,Medium_pH,Ionic_Strength,Temperature_C,Temperature_K,Interaction
0,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,8.231468,-0.486796,64489.4386,Positive,7.4,0.15,25.0,298.15,0
1,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,9.438832,-0.865882,27884.9954,Positive,7.4,0.15,25.0,298.15,0
2,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,5.557739,-0.862530,48197.1750,Negative,7.4,0.15,25.0,298.15,1
3,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,8.676558,-0.606215,100829.9852,Positive,7.4,0.15,25.0,298.15,0
4,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,5.265132,-0.157192,250533.5881,Negative,7.4,0.15,25.0,298.15,1



Combined Interaction DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8328093 entries, 0 to 8328092
Data columns (total 21 columns):
 #   Column            Dtype  
---  ------            -----  
 0   NPs               object 
 1   coresize          float64
 2   hydrosize         float64
 3   surfcharge        float64
 4   surfarea          float64
 5   Ec                float64
 6   Expotime          int64  
 7   dosage            float64
 8   e                 float64
 9   NOxygen           int64  
 10  class             object 
 11  protein_sequence  object 
 12  pI                float64
 13  GRAVY             float64
 14  MW                float64
 15  Charge_at_pH_7_4  object 
 16  Medium_pH         float64
 17  Ionic_Strength    float64
 18  Temperature_C     float64
 19  Temperature_K     float64
 20  Interaction       int64  
dtypes: float64(14), int64(3), object(4)
memory usage: 1.3+ GB


In [4]:
import os

# List files in the toxicity dataset directory to confirm file names
toxicity_files = os.listdir(toxicity_path)
print(f"Files in Nanoparticle Toxicity dataset directory: {toxicity_files}")

# Load Nanoparticle Toxicity Dataset with the correct file name
toxicity_df = pd.read_csv(f"{toxicity_path}/nanotox_dataset.csv")
print("\nNanoparticle Toxicity DataFrame Head:")
display(toxicity_df.head())
print("\nNanoparticle Toxicity DataFrame Info:")
toxicity_df.info()

Files in Nanoparticle Toxicity dataset directory: ['nanotox_dataset.csv', '.nfs000000001c7d55af00000074']

Nanoparticle Toxicity DataFrame Head:


,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,class
0,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,nonToxic
1,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.010,1.61,3,nonToxic
2,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.100,1.61,3,nonToxic
3,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,1.000,1.61,3,nonToxic
4,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,5.000,1.61,3,nonToxic



Nanoparticle Toxicity DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 881 entries, 0 to 880
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   NPs         881 non-null    object 
 1   coresize    881 non-null    float64
 2   hydrosize   881 non-null    float64
 3   surfcharge  881 non-null    float64
 4   surfarea    881 non-null    float64
 5   Ec          881 non-null    float64
 6   Expotime    881 non-null    int64  
 7   dosage      881 non-null    float64
 8   e           881 non-null    float64
 9   NOxygen     881 non-null    int64  
 10  class       881 non-null    object 
dtypes: float64(7), int64(2), object(2)
memory usage: 75.8+ KB


### Step 3: Generate Protein Physicochemical Descriptors

Using `BioPython`'s `SeqUtils` module to calculate the Isoelectric Point (pI), GRAVY, Molecular Weight (MW), and Charge at pH 7.4 for each unique protein sequence from the `ppi_df`.


In [5]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Seq import Seq

def get_protein_features(sequence):
    if not isinstance(sequence, str) or not sequence:
        return None, None, None, None
    try:
        # Ensure the sequence is a Biopython Seq object
        protein_seq = Seq(sequence)
        analysed_seq = ProteinAnalysis(str(protein_seq))

        # Isoelectric Point (pI)
        pI = analysed_seq.isoelectric_point()

        # GRAVY (Grand Average of Hydropathy)
        gravy = analysed_seq.gravy()

        # Molecular Weight (MW)
        mw = analysed_seq.molecular_weight()

        # Charge at pH 7.4 (approximate blood pH)
        # This is a simplified calculation, more precise methods exist but this serves as a good start.
        # The ProtParam.charge() method estimates net charge at a given pH, but it's not directly exposed
        # in a straightforward manner for a specific pH. For a precise calculation, one would typically
        # sum charges of all titratable groups at pH 7.4 using pKa values.
        # A common simplified approach is to use the pI and assume the protein is negatively charged
        # if pH > pI, and positively charged if pH < pI.
        # For a rough estimate of charge at pH 7.4 relative to pI:
        if pI > 7.4:
            charge_at_7_4 = 'Positive' # More basic, so more positive charge at physiological pH
        elif pI < 7.4:
            charge_at_7_4 = 'Negative' # More acidic, so more negative charge at physiological pH
        else:
            charge_at_7_4 = 'Neutral'

        # Note: A more rigorous calculation of charge at specific pH would involve
        # knowing the pKa values of each amino acid residue and summing up their
        # protonation states at the given pH. Biopython's ProtParam can give
        # amino acid composition, which can then be used with external pKa tables.
        # For this exercise, we will use the pI comparison for a qualitative charge.

        return pI, gravy, mw, charge_at_7_4
    except Exception as e:
        print(f"Error processing sequence {sequence[:20]}...: {e}")
        return None, None, None, None

# Get all unique protein sequences from both columns of ppi_df
unique_protein_sequences = pd.concat([ppi_df['protein_sequences_1'], ppi_df['protein_sequences_2']]).unique()

# Create a DataFrame to store protein features
protein_features = []
for seq in unique_protein_sequences:
    pI, gravy, mw, charge_7_4 = get_protein_features(seq)
    protein_features.append({'sequence': seq, 'pI': pI, 'GRAVY': gravy, 'MW': mw, 'Charge_at_pH_7_4': charge_7_4})

protein_features_df = pd.DataFrame(protein_features)
display(protein_features_df.head())
protein_features_df.info()

Error processing sequence MERQEESLSARPALETEGLR...: 'U'
Error processing sequence MGCAEGKAVAAAAPTELQTK...: 'U'
Error processing sequence MCAARLAAAAAAAQSVYAFS...: 'U'


,sequence,pI,GRAVY,MW,Charge_at_pH_7_4
0,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,8.231468,-0.486796,64489.4386,Positive
1,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,9.438832,-0.865882,27884.9954,Positive
2,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,5.557739,-0.862530,48197.1750,Negative
3,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,8.676558,-0.606215,100829.9852,Positive
4,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,5.265132,-0.157192,250533.5881,Negative


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9456 entries, 0 to 9455
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sequence          9456 non-null   object 
 1   pI                9453 non-null   float64
 2   GRAVY             9453 non-null   float64
 3   MW                9453 non-null   float64
 4   Charge_at_pH_7_4  9453 non-null   object 
dtypes: float64(3), object(2)
memory usage: 369.5+ KB


Step 4: Combine Datasets, Add Environmental Variables, and Generate Target Variable

 creating a comprehensive dataset by merging the nanoparticle toxicity data with the calculated protein features. As we are  looking for protein-nanoparticle interactions, we create a cartesian product (all possible pairs) between the unique nanoparticles and unique proteins. Then,add columns for standard experimental environmental conditions and generate a binary target variable ('Interaction') based on the provided logic.

In [6]:
# Clean protein_features_df by dropping rows with None values (failed calculations)
# and rename 'sequence' for clarity
protein_features_df_cleaned = protein_features_df.dropna(subset=['pI', 'GRAVY', 'MW', 'Charge_at_pH_7_4']).copy()
protein_features_df_cleaned = protein_features_df_cleaned.rename(columns={'sequence': 'protein_sequence'})

# To create a Cartesian product of nanoparticles and proteins, we'll add a dummy key to both dataframes
toxicity_df_temp = toxicity_df.copy()
protein_features_df_temp = protein_features_df_cleaned.copy()

toxicity_df_temp['key'] = 1
protein_features_df_temp['key'] = 1

# Perform a merge on the dummy key to get all combinations
interaction_df = pd.merge(toxicity_df_temp, protein_features_df_temp, on='key').drop('key', axis=1)

# Add Environmental Conditions (Standard Lab Constants)
# These are illustrative constant values as requested.
interaction_df['Medium_pH'] = 7.4
interaction_df['Ionic_Strength'] = 0.15 # Molar (e.g., for PBS)
interaction_df['Temperature_C'] = 25.0 # Celsius
interaction_df['Temperature_K'] = 298.15 # Kelvin

# Generate Target Variable ('Interaction' Binary) based on the specific logic:
# "If Nanoparticle Zeta Potential is Positive (from the Toxicity dataset)
# AND Protein pI is < 6 (meaning it's Negative at blood pH) THEN set the missing 'Interaction' label to 1."

# Initialize 'Interaction' column to 0 (no interaction by default)
interaction_df['Interaction'] = 0

# Apply the specific rule: if nanoparticle surfcharge > 0 and protein pI < 6, set Interaction to 1
interaction_df.loc[(interaction_df['surfcharge'] > 0) & (interaction_df['pI'] < 6), 'Interaction'] = 1

print("Combined Interaction DataFrame Head:")
display(interaction_df.head())
print("\nCombined Interaction DataFrame Info:")
interaction_df.info()

Combined Interaction DataFrame Head:


,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,...,protein_sequence,pI,GRAVY,MW,Charge_at_pH_7_4,Medium_pH,Ionic_Strength,Temperature_C,Temperature_K,Interaction
0,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,8.231468,-0.486796,64489.4386,Positive,7.4,0.15,25.0,298.15,0
1,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,9.438832,-0.865882,27884.9954,Positive,7.4,0.15,25.0,298.15,0
2,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,5.557739,-0.862530,48197.1750,Negative,7.4,0.15,25.0,298.15,1
3,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,8.676558,-0.606215,100829.9852,Positive,7.4,0.15,25.0,298.15,0
4,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,...,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,5.265132,-0.157192,250533.5881,Negative,7.4,0.15,25.0,298.15,1



Combined Interaction DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8328093 entries, 0 to 8328092
Data columns (total 21 columns):
 #   Column            Dtype  
---  ------            -----  
 0   NPs               object 
 1   coresize          float64
 2   hydrosize         float64
 3   surfcharge        float64
 4   surfarea          float64
 5   Ec                float64
 6   Expotime          int64  
 7   dosage            float64
 8   e                 float64
 9   NOxygen           int64  
 10  class             object 
 11  protein_sequence  object 
 12  pI                float64
 13  GRAVY             float64
 14  MW                float64
 15  Charge_at_pH_7_4  object 
 16  Medium_pH         float64
 17  Ionic_Strength    float64
 18  Temperature_C     float64
 19  Temperature_K     float64
 20  Interaction       int64  
dtypes: float64(14), int64(3), object(4)
memory usage: 1.3+ GB


### Step 5: Prepare Data for Machine Learning

prepare the `interaction_df` for machine learning. This involves selecting features, handling categorical variables, and splitting the data into training and testing sets. For categorical features
use one-hot encoding.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define features (X) and target (y)
X = interaction_df.drop('Interaction', axis=1)
y = interaction_df['Interaction']

# Identify categorical and numerical columns for preprocessing
# Exclude 'protein_sequence' as it's been converted to features (pI, GRAVY, MW, Charge_at_pH_7_4)
categorical_features = ['NPs', 'class', 'Charge_at_pH_7_4']
numerical_features = [
    'coresize', 'hydrosize', 'surfcharge', 'surfarea', 'Ec',
    'Expotime', 'dosage', 'e', 'NOxygen', 'pI', 'GRAVY', 'MW',
    'Medium_pH', 'Ionic_Strength', 'Temperature_C', 'Temperature_K'
]

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop' # Drop 'protein_sequence' and any other unlisted columns
)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (6662474, 20)
Shape of X_test: (1665619, 20)
Shape of y_train: (6662474,)
Shape of y_test: (1665619,)


### Step 6: Train the Machine Learning Model (SVM)
train a Support Vector Machine (SVM) classifier. Given the large dataset size, using a `LinearSVC` is more computationally efficient for large datasets compared to `SVC` with a non-linear kernel.  integrate the `preprocessor` into a `Pipeline` for streamlined training and prediction.

In [8]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# Create the SVM pipeline
# Due to the very large dataset, LinearSVC is chosen for efficiency.
# Ensure max_iter is sufficient for convergence
svm_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LinearSVC(random_state=42, dual=False, max_iter=1000))
    ]
)

print("Training the SVM model...")
# Train the model
svm_pipeline.fit(X_train, y_train)
print("SVM model training complete.")

Training the SVM model...
SVM model training complete.


### Step 7: Evaluate the Model

After training, evaluate the SVM model's performance on the test set using classification metrics like accuracy, precision, recall, and F1-score.

In [9]:
# Make predictions on the test set
y_pred = svm_pipeline.predict(X_test)

# Evaluate the model
print("\nModel Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Model Evaluation:
Accuracy: 0.9984
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1438348
           1       0.99      0.99      0.99    227271

    accuracy                           1.00   1665619
   macro avg       1.00      1.00      1.00   1665619
weighted avg       1.00      1.00      1.00   1665619



##CONCLUSION:
From the model evaluation, we can conclude that the SVM model performs exceptionally well in predicting nanoparticle-protein interactions.

Accuracy: With an accuracy of 0.9984 (99.84%), the model correctly predicts the interaction or non-interaction for almost all cases in the test set.

Classification Report:

For Class 0 (No Interaction): The precision, recall, and F1-score are all 1.00. This means the model is nearly perfect at identifying cases where no interaction occurs, with very few false positives or false negatives for this class.

For Class 1 (Interaction): The precision, recall, and F1-score are all 0.99. This indicates that the model is also highly effective at identifying actual interactions, with a very low rate of misclassifications for this class.

Overall, the model demonstrates strong predictive capability for this binary classification task.